In [26]:

import pandas as pd
from pathlib import Path

# --------------------------------------------------
# 1. Dateipfade definieren
# --------------------------------------------------

file_original = Path(
    r"C:\Users\incog\Desktop\Portfolio_Project_Agriculture\data\processed\ble_cleaned_supply_balance_with_quotas_1991-2025.xlsx"
)

file_NICHT_FAO = Path(
    r"C:\Users\incog\Desktop\Portfolio_Project_Agriculture\data\processed\ble_cleaned_supply_balance_with_quotas_NICHT_FAO_1991-2025.xlsx"
)

# --------------------------------------------------
# 2. Prüfen, ob beide Dateien existieren
# --------------------------------------------------

for file in [file_original, file_NICHT_FAO]:
    if not file.exists():
        raise FileNotFoundError(f"Datei nicht gefunden: {file}")

print("Beide Exceldateien wurden gefunden.")

# --------------------------------------------------
# 3. Exceldateien einlesen
# --------------------------------------------------

excel_original = pd.ExcelFile(file_original)
excel_NICHT_FAO = pd.ExcelFile(file_NICHT_FAO)

print("\nTabellenblätter – Originaldatei:")
print(excel_original.sheet_names)

print("\nTabellenblätter – Datei NICHT_FAO:")
print(excel_NICHT_FAO.sheet_names)

# --------------------------------------------------
# 4. Tabellenblätter vergleichen
# --------------------------------------------------

sheets_original = set(excel_original.sheet_names)
sheets_NICHT_FAO = set(excel_NICHT_FAO.sheet_names)

if sheets_original == sheets_NICHT_FAO:
    print("\nIDENTISCHE Tabellenblattnamen.")
else:
    print("\nUNTERSCHIEDE bei den Tabellenblättern:")

    print(
        "Nur in der Originaldatei:",
        sorted(sheets_original - sheets_NICHT_FAO)
    )

    print(
        "Nur in der Datei NICHT_FAO:",
        sorted(sheets_NICHT_FAO - sheets_original)
    )

# --------------------------------------------------
# 5. Gemeinsame Tabellenblätter anzeigen
# --------------------------------------------------

common_sheets = sorted(sheets_original & sheets_NICHT_FAO)

print("\nGemeinsame Tabellenblätter:")
for sheet in common_sheets:
    print(f"- {sheet}")

print(f"\nAnzahl Tabellenblätter Original: {len(sheets_original)}")
print(f"Anzahl Tabellenblätter NICHT_FAO: {len(sheets_NICHT_FAO)}")
print(f"Anzahl gemeinsamer Tabellenblätter: {len(common_sheets)}")

# --------------------------------------------------
# 6. Exceldateien schließen
# --------------------------------------------------

excel_original.close()
excel_NICHT_FAO.close()

print("\nVergleich der Tabellenblattnamen abgeschlossen.")


Beide Exceldateien wurden gefunden.

Tabellenblätter – Originaldatei:
['Sheet1']

Tabellenblätter – Datei NICHT_FAO:
['Sheet1']

IDENTISCHE Tabellenblattnamen.

Gemeinsame Tabellenblätter:
- Sheet1

Anzahl Tabellenblätter Original: 1
Anzahl Tabellenblätter NICHT_FAO: 1
Anzahl gemeinsamer Tabellenblätter: 1

Vergleich der Tabellenblattnamen abgeschlossen.


In [27]:

# --------------------------------------------------
# 5. Tabellenwerte korrekt vergleichen
# --------------------------------------------------

import numpy as np

differences = []
summary = []

common_sheets = [
    sheet for sheet in excel_original.sheet_names
    if sheet in excel_NICHT_FAO.sheet_names
]

for sheet in common_sheets:

    df_NICHT_FAO = pd.read_excel(
        file_NICHT_FAO,
        sheet_name=sheet
    )

    df_original = pd.read_excel(
        file_original,
        sheet_name=sheet
    )

    # Zeilen- und Spaltenanzahl
    rows_n, cols_n = df_NICHT_FAO.shape
    rows_o, cols_o = df_original.shape

    print(f"\nTabellenblatt: {sheet}")
    print(f"NICHT_FAO: {df_NICHT_FAO.shape}")
    print(f"Original:  {df_original.shape}")

    # Spaltennamen und Reihenfolge vergleichen
    columns_equal = df_NICHT_FAO.columns.equals(
        df_original.columns
    )

    # Abweichungen bei Spaltennamen dokumentieren
    max_cols = max(cols_n, cols_o)
    max_rows = max(rows_n, rows_o)

    # Vergleich anhand tatsächlicher Spaltenpositionen
    for col_idx in range(max_cols):

        if col_idx < cols_n:
            col_n = df_NICHT_FAO.columns[col_idx]
        else:
            col_n = "<Spalte fehlt>"

        if col_idx < cols_o:
            col_o = df_original.columns[col_idx]
        else:
            col_o = "<Spalte fehlt>"

        # Spalte existiert nur in einer Datei
        if col_idx >= cols_n or col_idx >= cols_o:

            for row_idx in range(max_rows):

                val_n = (
                    df_NICHT_FAO.iloc[row_idx, col_idx]
                    if row_idx < rows_n and col_idx < cols_n
                    else np.nan
                )

                val_o = (
                    df_original.iloc[row_idx, col_idx]
                    if row_idx < rows_o and col_idx < cols_o
                    else np.nan
                )

                # Nur tatsächlich vorhandene Werte dokumentieren
                if pd.notna(val_n) or pd.notna(val_o):
                    differences.append({
                        "Tabellenblatt": sheet,
                        "Excel-Zeile": row_idx + 2,
                        "Spaltenposition": col_idx + 1,
                        "Spalte_NICHT_FAO": col_n,
                        "Spalte_Original": col_o,
                        "Wert_NICHT_FAO": val_n,
                        "Wert_Original": val_o,
                        "Abweichungstyp": "Zusätzliche/fehlende Spalte"
                    })

            continue

        # Beide Spalten existieren:
        # Werte zeilenweise vergleichen
        for row_idx in range(max_rows):

            val_n = (
                df_NICHT_FAO.iloc[row_idx, col_idx]
                if row_idx < rows_n
                else np.nan
            )

            val_o = (
                df_original.iloc[row_idx, col_idx]
                if row_idx < rows_o
                else np.nan
            )

            # Fehlende Werte auf beiden Seiten gelten als identisch
            if pd.isna(val_n) and pd.isna(val_o):
                continue

            # Unterschiedliche Werte erkennen
            try:
                equal = val_n == val_o
                if pd.isna(equal):
                    equal = False
            except (TypeError, ValueError):
                equal = False

            if not equal:
                differences.append({
                    "Tabellenblatt": sheet,
                    "Excel-Zeile": row_idx + 2,
                    "Spaltenposition": col_idx + 1,
                    "Spalte_NICHT_FAO": col_n,
                    "Spalte_Original": col_o,
                    "Wert_NICHT_FAO": val_n,
                    "Wert_Original": val_o,
                    "Abweichungstyp": "Unterschiedlicher Zellwert"
                })

    # Zusammenfassung
    sheet_differences = [
        d for d in differences
        if d["Tabellenblatt"] == sheet
    ]

    summary.append({
        "Tabellenblatt": sheet,
        "Zeilen_NICHT_FAO": rows_n,
        "Zeilen_Original": rows_o,
        "Spalten_NICHT_FAO": cols_n,
        "Spalten_Original": cols_o,
        "Spaltennamen_identisch": columns_equal,
        "Anzahl_abweichender_Zellen": len(sheet_differences),
        "Werte_identisch": len(sheet_differences) == 0
    })


# --------------------------------------------------
# 6. Ergebnisse ausgeben
# --------------------------------------------------

summary_df = pd.DataFrame(summary)
differences_df = pd.DataFrame(differences)

print("\n" + "=" * 70)
print("GESAMTERGEBNIS: NICHT_FAO VS. ORIGINAL")
print("=" * 70)

if summary_df.empty:
    print("Keine gemeinsamen Tabellenblätter gefunden.")
else:
    print(summary_df.to_string(index=False))

if differences_df.empty:
    print("\nKeine Abweichungen gefunden.")
else:
    print(
        f"\nAnzahl dokumentierter Abweichungen: "
        f"{len(differences_df)}"
    )
    print("\nErste 20 Abweichungen:")
    print(differences_df.head(20).to_string(index=False))


# --------------------------------------------------
# 7. Ergebnisse speichern
# --------------------------------------------------

output_comparison = (
    file_NICHT_FAO.parent
    / "vergleich_NICHT_FAO_vs_original.xlsx"
)

with pd.ExcelWriter(
    output_comparison,
    engine="openpyxl"
) as writer:

    summary_df.to_excel(
        writer,
        sheet_name="Zusammenfassung",
        index=False
    )

    differences_df.to_excel(
        writer,
        sheet_name="Abweichungen",
        index=False
    )

print(f"\nVergleich gespeichert unter:\n{output_comparison}")



Tabellenblatt: Sheet1
NICHT_FAO: (102, 41)
Original:  (102, 4)

GESAMTERGEBNIS: NICHT_FAO VS. ORIGINAL
Tabellenblatt  Zeilen_NICHT_FAO  Zeilen_Original  Spalten_NICHT_FAO  Spalten_Original  Spaltennamen_identisch  Anzahl_abweichender_Zellen  Werte_identisch
       Sheet1               102              102                 41                 4                   False                        2696            False

Anzahl dokumentierter Abweichungen: 2696

Erste 20 Abweichungen:
Tabellenblatt  Excel-Zeile  Spaltenposition Spalte_NICHT_FAO Spalte_Original Wert_NICHT_FAO Wert_Original             Abweichungstyp
       Sheet1            3                1  Wirtschaftsjahr Wirtschaftsjahr        1991/92       1992/93 Unterschiedlicher Zellwert
       Sheet1            4                1  Wirtschaftsjahr Wirtschaftsjahr        1991/92       1993/94 Unterschiedlicher Zellwert
       Sheet1            5                1  Wirtschaftsjahr Wirtschaftsjahr        1992/93       1994/95 Unterschiedlich

In [19]:

import pandas as pd
from pathlib import Path
from collections import Counter
import numpy as np

# --------------------------------------------------
# 1. Dateipfade definieren
# --------------------------------------------------

file_original = Path(
    r"C:\Users\incog\Desktop\Portfolio_Project_Agriculture\data\processed\ble_cleaned_supply_balance_with_quotas_1991-2025.xlsx"
)

file_heute = Path(
    r"C:\Users\incog\Desktop\Portfolio_Project_Agriculture\data\processed\ble_cleaned_supply_balance_with_quotas_NICHT_FAO_1991-2025.xlsx"
)

# --------------------------------------------------
# 2. Prüfen, ob beide Dateien existieren
# --------------------------------------------------

for file in [file_heute, file_original]:
    if not file.exists():
        raise FileNotFoundError(
            f"Datei nicht gefunden: {file}"
        )

print("Beide Exceldateien wurden gefunden.")

# --------------------------------------------------
# 3. Alle Tabellenblätter einlesen
# --------------------------------------------------

data_heute = pd.read_excel(
    file_heute,
    sheet_name=None
)

data_original = pd.read_excel(
    file_original,
    sheet_name=None
)

print("\nTabellenblätter – Datei HEUTE:")
print(list(data_heute.keys()))

print("\nTabellenblätter – Originaldatei:")
print(list(data_original.keys()))

# --------------------------------------------------
# 4. Hilfsfunktion zum Normalisieren von Zellwerten
# --------------------------------------------------

def normalize_value(value):
    """Vereinheitlicht fehlende Werte und vergleichbare Datentypen."""

    if pd.isna(value):
        return ("MISSING",)

    # NumPy-Datentypen in Python-Datentypen umwandeln
    if isinstance(value, np.generic):
        value = value.item()

    # Zahlen konsistent behandeln, aber bool nicht als Zahl werten
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return ("NUMBER", float(value))

    # Andere Datentypen einschließlich Text und bool erhalten
    return (type(value).__name__, repr(value))


def row_counter(df):
    """Zählt Zeilen unabhängig von ihrer Reihenfolge."""

    return Counter(
        tuple(normalize_value(value) for value in row)
        for row in df.itertuples(index=False, name=None)
    )

# --------------------------------------------------
# 5. Tabellenblattnamen vergleichen
# --------------------------------------------------

sheets_heute = set(data_heute.keys())
sheets_original = set(data_original.keys())

nur_heute = sheets_heute - sheets_original
nur_original = sheets_original - sheets_heute

if nur_heute or nur_original:
    print("\nUNTERSCHIEDE BEI DEN TABELLENBLÄTTERN:")

    print("Nur in HEUTE:", sorted(nur_heute))
    print("Nur im Original:", sorted(nur_original))
else:
    print("\nIDENTISCHE TABELLENBLATTNAMEN.")

# --------------------------------------------------
# 6. Tabellenwerte vergleichen
# --------------------------------------------------

alle_identisch = not (nur_heute or nur_original)

for sheet in data_heute:

    # Nur gemeinsame Tabellenblätter vergleichen
    if sheet not in data_original:
        continue

    df1 = data_heute[sheet]
    df2 = data_original[sheet]

    print("\n" + "=" * 70)
    print(f"TABELLENBLATT: {sheet}")
    print("=" * 70)

    print(f"Zeilen HEUTE:       {len(df1)}")
    print(f"Zeilen Original:    {len(df2)}")
    print(f"Spalten HEUTE:      {len(df1.columns)}")
    print(f"Spalten Original:   {len(df2.columns)}")

    # Spaltennamen und Reihenfolge prüfen
    if not df1.columns.equals(df2.columns):
        print("\nERGEBNIS: Spaltennamen oder Spaltenreihenfolge unterscheiden sich.")
        print("Spalten HEUTE:", df1.columns.tolist())
        print("Spalten Original:", df2.columns.tolist())

        alle_identisch = False
        continue

    # Zeilen unabhängig von ihrer Reihenfolge vergleichen
    rows1 = row_counter(df1)
    rows2 = row_counter(df2)

    if rows1 == rows2:
        print(
            "\nERGEBNIS: Alle Zeilenwerte sind identisch."
        )
        print(
            "Die Reihenfolge der Zeilen spielt keine Rolle."
        )

    else:
        alle_identisch = False

        print("\nERGEBNIS: Unterschiede gefunden.")

        nur_in_heute = rows1 - rows2
        nur_im_original = rows2 - rows1

        print("\nZeilen/Datensätze nur in HEUTE:")

        if nur_in_heute:
            for row, count in nur_in_heute.items():
                print(f"{count}x {row}")
        else:
            print("Keine")

        print("\nZeilen/Datensätze nur im Original:")

        if nur_im_original:
            for row, count in nur_im_original.items():
                print(f"{count}x {row}")
        else:
            print("Keine")

# --------------------------------------------------
# 7. Gesamtergebnis
# --------------------------------------------------

print("\n" + "=" * 70)
print("GESAMTERGEBNIS")
print("=" * 70)

if alle_identisch:
    print("Alle Tabellenwerte sind identisch.")
    print("Die Zeilenreihenfolge darf unterschiedlich sein.")
else:
    print("Die Tabellen unterscheiden sich.")


Beide Exceldateien wurden gefunden.

Tabellenblätter – Datei HEUTE:
['Sheet1']

Tabellenblätter – Originaldatei:
['Sheet1']

IDENTISCHE TABELLENBLATTNAMEN.

TABELLENBLATT: Sheet1
Zeilen HEUTE:       102
Zeilen Original:    102
Spalten HEUTE:      41
Spalten Original:   28

ERGEBNIS: Spaltennamen oder Spaltenreihenfolge unterscheiden sich.
Spalten HEUTE: ['Wirtschaftsjahr', 'Bilanz Produkt', 'Ausfuhr', 'Einfuhr', 'Inlandsverwendung', 'Inlandsverwendung insgesamt', 'Selbstversorgungsgrad in %', 'Verwendbare Erzeugung', 'Inlandsverwendung standardisiert', 'Importquote in %', 'Exportquote in %', 'Futter_x', 'Industrielle Verwertung_x', 'Industrielle Verwertung für Energie_x', 'Nahrungsverbrauch_x', 'Nahrungsverbrauch Ausbeute in %_x', 'Nahrungsverbrauch Kartoffelerzeugnisse_x', 'Nahrungsverbrauch in Getreidewert_x', 'Nahrungsverbrauch in Getreidewert in kg je Kopf_x', 'Nahrungsverbrauch in Produktgewicht_x', 'Nahrungsverbrauch in Produktgewicht in kg je Kopf_x', 'Nahrungsverbrauch in Produ

In [20]:

import pandas as pd
from pathlib import Path

# --------------------------------------------------
# 1. Dateien einlesen
# --------------------------------------------------

df_heute = pd.read_excel(file_heute)
df_original = pd.read_excel(file_original)

# --------------------------------------------------
# 2. Spalten vergleichen
# --------------------------------------------------

cols_heute = set(df_heute.columns)
cols_original = set(df_original.columns)

print("=" * 70)
print("SPALTENANALYSE")
print("=" * 70)

print(f"\nAnzahl Spalten HEUTE: {len(df_heute.columns)}")
print(f"Anzahl Spalten Original: {len(df_original.columns)}")

print("\nSpalten nur in HEUTE:")
for col in df_heute.columns:
    if col not in cols_original:
        print(" -", col)

print("\nSpalten nur im Original:")
for col in df_original.columns:
    if col not in cols_heute:
        print(" -", col)

# --------------------------------------------------
# 3. Prüfen, ob _x- und _y-Spalten identisch sind
# --------------------------------------------------

print("\n" + "=" * 70)
print("PRÜFUNG DOPPELTER SPALTEN")
print("=" * 70)

for col in df_heute.columns:
    if col.endswith("_x"):
        base = col[:-2]
        col_y = base + "_y"

        if col_y in df_heute.columns:
            identisch = df_heute[col].equals(df_heute[col_y])
            print(
                f"{base}: _x und _y identisch? {identisch}"
            )

# --------------------------------------------------
# 4. Prüfen, ob wichtige Originalspalten fehlen
# --------------------------------------------------

wichtige_spalten = [
    "Anfangsbestand",
    "Bestandsänderung",
    "Endbestand",
    "Gesamterzeugung",
    "Futter",
    "Industrielle Verwertung",
    "Verluste"
]

print("\n" + "=" * 70)
print("PRÜFUNG WICHTIGER ORIGINALSPALTEN")
print("=" * 70)

for col in wichtige_spalten:
    print(
        f"{col}: "
        f"HEUTE={'vorhanden' if col in df_heute.columns else 'FEHLT'}, "
        f"Original={'vorhanden' if col in df_original.columns else 'FEHLT'}"
    )


SPALTENANALYSE

Anzahl Spalten HEUTE: 41
Anzahl Spalten Original: 28

Spalten nur in HEUTE:
 - Importquote in %
 - Exportquote in %
 - Futter_x
 - Industrielle Verwertung_x
 - Industrielle Verwertung für Energie_x
 - Nahrungsverbrauch_x
 - Nahrungsverbrauch Ausbeute in %_x
 - Nahrungsverbrauch Kartoffelerzeugnisse_x
 - Nahrungsverbrauch in Getreidewert_x
 - Nahrungsverbrauch in Getreidewert in kg je Kopf_x
 - Nahrungsverbrauch in Produktgewicht_x
 - Nahrungsverbrauch in Produktgewicht in kg je Kopf_x
 - Nahrungsverbrauch in Produktgewicht in kg je Kopf/ dgl. kg je Kopf_x
 - Nahrungsverbrauch in Produktgewicht/ in Mehlwert_x
 - Nahrungsverbrauch in kg je Kopf_x
 - Saatgut_x
 - Verluste_x
 - Futter_y
 - Industrielle Verwertung_y
 - Industrielle Verwertung für Energie_y
 - Nahrungsverbrauch_y
 - Nahrungsverbrauch Ausbeute in %_y
 - Nahrungsverbrauch Kartoffelerzeugnisse_y
 - Nahrungsverbrauch in Getreidewert_y
 - Nahrungsverbrauch in Getreidewert in kg je Kopf_y
 - Nahrungsverbrauch in Pr

In [11]:
# Exceldateien schließen
excel_heute.close()
excel_original.close()

print("Beide Exceldateien wurden geschlossen.")

Beide Exceldateien wurden geschlossen.
